## 1. Dependencias y herramientas base

In [ ]:
!sudo apt install -y git curl python3 build-essential bc bison flex libssl-dev libncurses-dev
!sudo curl -o /usr/local/bin/repo https://storage.googleapis.com/git-repo-downloads/repo
!sudo chmod a+x /usr/local/bin/repo

!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"


## 2. Descargar el árbol del kernel (repo sync)

In [ ]:
!mkdir -p ~/lamu-kernel
%cd ~/lamu-kernel

!yes | repo init -u https://github.com/moto-mediatek-devs/android_kernel_motorola_lamu_manifest -b lineage-23.2 --depth=1
!repo sync -c -j24 --force-sync --no-clone-bundle --optimized-fetch

## 3. Parche kABI obligatorio (kernel < 6.12)

Sin este parche, habilitar `CONFIG_SYSVIPC`/`CONFIG_IPC_NS` desplaza offsets del `task_struct` y bootloopea por romper kABI. Confirmado inocuo por sí solo (probado con defconfig stock, arrancó bien).

In [ ]:
%cd ~/lamu-kernel
!wget -O kabi_sysvipc.patch https://raw.githubusercontent.com/ravindu644/Droidspaces-OSS/main/Documentation/resources/kernel-patches/GKI/below-kernel-6.12/001.GKI-below-6.12-fix_sysvipc_kabi_6_7_8.patch

%cd ~/lamu-kernel/kernel-6.6
!patch -p1 < ~/lamu-kernel/kabi_sysvipc.patch
!rm -f include/linux/sched.h.orig


## 4. Editar `gki_defconfig`

Opciones del bloque "GKI-Exclusive Configuration" de la doc de Droidspaces. El `gki_defconfig` stock no trae ninguna, salvo `# CONFIG_PID_NS is not set`.

- `SYSVIPC`, `POSIX_MQUEUE`, `NETFILTER_XT_MATCH_ADDRTYPE` y `DEVTMPFS` se insertan en su **posición canónica**. Si quedan al final del archivo, Kleaf falla con `savedefconfig does not match`.
- `PID_NS` e `IPC_NS` **no** se escriben explícitamente: basta con borrar `# CONFIG_PID_NS is not set` y ambas resuelven en `=y` por defecto. Si se escriben, `savedefconfig` las marca como redundantes. Se confirma en el paso 6.
- Las opciones de Docker Swarm (`VXLAN`, `IP_VS`, `IP_VS_RR`, `BRIDGE_NETFILTER`) no se agregan: no están en la doc de Droidspaces y arriesgan kABI.

La celda es idempotente y **commitea el resultado**, para que el build tenga un hash de git único y distinguible de builds anteriores con otro defconfig.


In [ ]:
%%bash
set -e
cd ~/lamu-kernel/kernel-6.6
CFG=arch/arm64/configs/gki_defconfig

# 1. Borrar cualquier rastro previo de las opciones (=y o "is not set"),
#    así la celda se puede re-ejecutar sin duplicar líneas.
for opt in SYSVIPC POSIX_MQUEUE IPC_NS PID_NS DEVTMPFS NETFILTER_XT_MATCH_ADDRTYPE; do
  sed -i "/^CONFIG_${opt}=/d;/^# CONFIG_${opt} is not set\$/d" "$CFG"
done

# 2. Insertar en la posición canónica que exige savedefconfig.
#    Si el ancla no existe (otra rama del kernel), fallar en vez de agregar a ciegas.
insert_after() {
  grep -qxF "$1" "$CFG" || { echo "ERROR: ancla '$1' no encontrada en $CFG"; exit 1; }
  sed -i "/^$1\$/a $2" "$CFG"
}
insert_after 'CONFIG_LOCALVERSION="-4k"'           'CONFIG_SYSVIPC=y\nCONFIG_POSIX_MQUEUE=y'
insert_after 'CONFIG_NETFILTER_XT_TARGET_TCPMSS=y' 'CONFIG_NETFILTER_XT_MATCH_ADDRTYPE=y'
insert_after 'CONFIG_PCI_ENDPOINT=y'               'CONFIG_DEVTMPFS=y'

echo "--- Resultado ---"
grep -nE "CONFIG_(SYSVIPC|POSIX_MQUEUE|IPC_NS|PID_NS|DEVTMPFS|NETFILTER_XT_MATCH_ADDRTYPE)[= ]" "$CFG"

git add "$CFG"
git commit -m "Droidspaces GKI config en posicion canonica (savedefconfig)" || echo "(nada que commitear)"
git rev-parse --short HEAD


## 5. Compilar

In [ ]:
%cd ~/lamu-kernel
!./build_mgk_64_k66.sh


## 6. Verificación decisiva: config REAL embebido en el kernel compilado

Antes de tocar boot.img. Confirmar que las opciones estén en `=y` (incluidas `PID_NS` e `IPC_NS`, que resuelven por dependencia) y que las de Docker Swarm sigan desactivadas.

El hash de la versión (`-g<hash>`) es el que hay que buscar después en `uname`/pstore del teléfono para confirmar que arrancó este build.


In [ ]:
%%bash
cd ~/lamu-kernel
kernel-6.6/scripts/extract-ikconfig out/mgk_64_k66/dist/Image.gz | \
  grep -E "CONFIG_(SYSVIPC|PID_NS|IPC_NS|POSIX_MQUEUE|DEVTMPFS|NETFILTER_XT_MATCH_ADDRTYPE|BRIDGE_NETFILTER|VXLAN|IP_VS)[= ]"

echo "--- versión del kernel y git HEAD (deben coincidir en el hash) ---"
zcat out/mgk_64_k66/dist/Image.gz | strings | grep "Linux version" | head -1
git -C kernel-6.6 rev-parse --short HEAD


## 7. Empaquetar boot.img

`ramdisk size: 0` es normal en este dispositivo GKI (el ramdisk real vive en `vendor_boot.img`).

In [18]:
%cd ~/lamu-kernel
!wget -O boot.img https://mirrorbits.lineageos.org/full/lamu/20260806/boot.img

!python3 tools/mkbootimg/unpack_bootimg.py --boot_img boot.img --out unpacked
!cp out/mgk_64_k66/dist/Image.gz unpacked/kernel

!python3 tools/mkbootimg/mkbootimg.py \
  --header_version 4 \
  --os_version 16.0.0 \
  --os_patch_level 2026-07 \
  --kernel unpacked/kernel \
  --ramdisk unpacked/ramdisk \
  --cmdline "" \
  --output boot_new.img

!./prebuilts/kernel-build-tools/linux-x86/bin/avbtool add_hash_footer \
  --image boot_new.img \
  --partition_size 67108864 \
  --partition_name boot \
  --algorithm NONE


/root/lamu-kernel
--2026-08-07 02:25:00--  https://mirrorbits.lineageos.org/full/lamu/20260806/boot.img
Resolving mirrorbits.lineageos.org (mirrorbits.lineageos.org)... 116.202.242.188
Connecting to mirrorbits.lineageos.org (mirrorbits.lineageos.org)|116.202.242.188|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://mirrors.ocf.berkeley.edu/lineageos/full/lamu/20260806/boot.img [following]
--2026-08-07 02:25:00--  https://mirrors.ocf.berkeley.edu/lineageos/full/lamu/20260806/boot.img
Resolving mirrors.ocf.berkeley.edu (mirrors.ocf.berkeley.edu)... 169.229.200.70, 2607:f140:0:32::70
Connecting to mirrors.ocf.berkeley.edu (mirrors.ocf.berkeley.edu)|169.229.200.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67108864 (64M) [application/octet-stream]
Saving to: ‘boot.img’

boot.img            100%[===================>]  64.00M  20.0MB/s    in 4.0s    

2026-08-07 02:25:05 (16.1 MB/s) - ‘boot.img’ saved [67108864/67108864]

bo

## 8. Verificar el paquete final antes de descargar

In [19]:
%cd ~/lamu-kernel
!python3 tools/mkbootimg/unpack_bootimg.py --boot_img boot_new.img --out verify
!sha1sum verify/kernel out/mgk_64_k66/dist/Image.gz
# Los dos hashes DEBEN coincidir. Si no coinciden, no descargar/flashear todavia.


/root/lamu-kernel
boot magic: ANDROID!
kernel_size: 14862102
ramdisk size: 0
os version: 16.0.0
os patch level: 2026-07
boot image header version: 4
command line args: 
boot.img signature size: 0
b2da1736378d093856f7bcae6ba655a1ce31239b  verify/kernel
b2da1736378d093856f7bcae6ba655a1ce31239b  out/mgk_64_k66/dist/Image.gz


In [20]:
from google.colab import files
files.download("boot_new.img")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>